In [96]:
# CELL 1: Setup and load v3 annotations as v4 input

import pandas as pd
from pathlib import Path

BASE    = Path("/Users/davekokel/Projects/carp_v2")
WORKING = BASE / "seed_kits" / "legacy_wrangling" / "working"

OUT_SUFFIX = "_v5"

def out_csv(name: str) -> Path:
    return WORKING / f"{name}{OUT_SUFFIX}.csv"

auto_v3_path = WORKING / "imaging_roi_annotations_AUTO_v3.csv"
df = pd.read_csv(auto_v3_path)

df.shape, df.columns.tolist()[:20]

((976, 32),
 ['roi_dir',
  'date_experiment',
  'fish',
  'roi_name',
  'date_born',
  'parent_female',
  'parent_male',
  'genotype_base_codes',
  'genotype_allele_codes',
  'genotype_pretty',
  'genotype_marker_fluor_codes',
  'genotype_marker_tag_codes',
  'treatment_plasmid_base_codes',
  'treatment_rna_base_codes',
  'treatment_marker_fluor_codes',
  'treatment_marker_tag_codes',
  'all_marker_fluor_codes',
  'additional plasmids injected',
  'additional mRNAs injected',
  'additonal proteins injected'])

In [97]:
# CELL: Slug-level inference of parents and treatments from v5 (no mapper)

import pandas as pd
import math
import re

def is_blank(val: object) -> bool:
    if val is None:
        return True
    if isinstance(val, float) and math.isnan(val):
        return True
    return str(val).strip() == ""

# ───────── define experiment slug from roi_dir ─────────
# We take Foundation/Experiment, e.g.:
#   /.../Korra_Foundation/20250522_skittlez/fish1/roi1
# → slug = 'Korra_Foundation/20250522_skittlez'

def extract_slug(path: str) -> str:
    p = path.replace("\\", "/").rstrip("/")
    parts = p.split("/")
    if len(parts) >= 4:
        foundation = parts[-4]
        experiment = parts[-3]
        return f"{foundation}/{experiment}"
    return p

df["slug_infer"] = df["roi_dir"].astype(str).apply(extract_slug)

print("Number of inferred slugs:", df["slug_infer"].nunique())

# ───────── slug-wise inference ─────────

# we'll collect some debug info
slug_debug = []

for slug, sub in df.groupby("slug_infer"):
    n = len(sub)

    # parents: candidate values within this slug
    pf_non = sub["parent_female"].dropna().astype(str).str.strip()
    pm_non = sub["parent_male"].dropna().astype(str).str.strip()

    # we'll only infer parents if we have at least one non-blank value for both female and male
    can_infer_parents = (pf_non[pf_non != ""].size > 0) and (pm_non[pm_non != ""].size > 0)
    parent_female_cand = None
    parent_male_cand = None
    if can_infer_parents:
        # use the most frequent (mode) for robustness
        parent_female_cand = pf_non[pf_non != ""].mode().iloc[0]
        parent_male_cand   = pm_non[pm_non != ""].mode().iloc[0]

    # treatments: plasmid and RNA base codes
    tp_non = sub["treatment_plasmid_base_codes"].dropna().astype(str).str.strip()
    tr_non = sub["treatment_rna_base_codes"].dropna().astype(str).str.strip()

    can_infer_tp = tp_non[tp_non != ""].size > 0
    can_infer_tr = tr_non[tr_non != ""].size > 0

    # if there are multiple distinct treatment codes in a slug, we log it but still use a mode
    tp_cand = None
    tr_cand = None
    if can_infer_tp:
        tp_cand = tp_non[tp_non != ""].mode().iloc[0]
    if can_infer_tr:
        tr_cand = tr_non[tr_non != ""].mode().iloc[0]

    # Now, fill holes, but *only* where fields are blank

    # parents
    if can_infer_parents:
        mask_parent_holes = (
            (df["slug_infer"] == slug)
            & df["parent_female"].apply(is_blank)
            & df["parent_male"].apply(is_blank)
        )
        df.loc[mask_parent_holes, "parent_female"] = parent_female_cand
        df.loc[mask_parent_holes, "parent_male"]   = parent_male_cand
        n_parent_filled = mask_parent_holes.sum()
    else:
        n_parent_filled = 0

    # treatment plasmid base codes
    if can_infer_tp:
        mask_tp_holes = (
            (df["slug_infer"] == slug)
            & df["treatment_plasmid_base_codes"].apply(is_blank)
        )
        df.loc[mask_tp_holes, "treatment_plasmid_base_codes"] = tp_cand
        n_tp_filled = mask_tp_holes.sum()
    else:
        n_tp_filled = 0

    # treatment RNA base codes
    if can_infer_tr:
        mask_tr_holes = (
            (df["slug_infer"] == slug)
            & df["treatment_rna_base_codes"].apply(is_blank)
        )
        df.loc[mask_tr_holes, "treatment_rna_base_codes"] = tr_cand
        n_tr_filled = mask_tr_holes.sum()
    else:
        n_tr_filled = 0

    slug_debug.append(
        {
            "slug": slug,
            "n_rows": n,
            "can_infer_parents": can_infer_parents,
            "parent_female_cand": parent_female_cand,
            "parent_male_cand": parent_male_cand,
            "n_parent_filled": n_parent_filled,
            "can_infer_tp": can_infer_tp,
            "tp_cand": tp_cand,
            "n_tp_filled": n_tp_filled,
            "can_infer_tr": can_infer_tr,
            "tr_cand": tr_cand,
            "n_tr_filled": n_tr_filled,
        }
    )

# show some debug info for skittles / mem_* slugs
slug_debug_df = pd.DataFrame(slug_debug)
print("\n[SLUG-INFER] Summary for key slugs:")
display(
    slug_debug_df[
        slug_debug_df["slug"].str.contains("skittle|mem_histone|mem-histone|mem_mito", case=False, na=False)
    ]
)

Number of inferred slugs: 16

[SLUG-INFER] Summary for key slugs:


,slug,n_rows,can_infer_parents,parent_female_cand,parent_male_cand,n_parent_filled,can_infer_tp,tp_cand,n_tp_filled,can_infer_tr,tr_cand,n_tr_filled
2,Korra_Foundation/20250428_mem_histone,2,True,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),0,False,None,0,True,pDQM117,0
4,Korra_Foundation/20250513_skittles,44,False,None,None,0,False,None,0,False,None,0
5,Korra_Foundation/20250520_mem_histone,4,True,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),0,False,None,0,True,pDQM117,0
6,Korra_Foundation/20250521_skittles_no-membrane,5,True,Dennis (F2 of allele 318),Abe (F2 of allele 309),0,False,None,0,False,None,0
7,Korra_Foundation/20250522_skittlez,19,False,None,None,0,False,None,0,False,None,0
8,Korra_Foundation/20250523_mem_histone,5,True,ef1a:2xLynk:tdmSG(J) (F2 of allele 302),ef1a:2xLynk:tdmSG(J) (F2 of allele 302),0,False,None,0,True,pDQM117,0
9,Korra_Foundation/20250528_mem-histone,3,True,ef1a:2xLynk:tdmSG(J) (F2 of allele 302),ef1a:2xLynk:tdmSG(J) (F2 of allele 302),0,False,None,0,True,pDQM117,0
10,Korra_Foundation/20250528_skittlez,2,False,None,None,0,False,None,0,False,None,0


In [98]:
# CELL 2: Load constructs and tags for fusion/localization mapping (v5)

AUTOLOAD = BASE / "seed_kits" / "2025-11-15-121231-autoload"

constructs_path = AUTOLOAD / "constructs_plasmid.csv"
tags_path       = AUTOLOAD / "tags.xlsx"  # adjust if in a different kit

df_constructs = pd.read_csv(constructs_path)
df_tags       = pd.read_excel(tags_path)

print("df_constructs:", df_constructs.shape)
print("df_tags:", df_tags.shape)

df_constructs.head(), df_tags.head()

df_constructs: (286, 12)
df_tags: (10, 5)


(  plasmid_code            plasmid_name        plasmid_nickname resistance  \
 0      pDQM001  CMV-SP6-mSG(J) IDT opt  CMV-SP6-mSG(J) IDT opt        Amp   
 1      pDQM002  CMV-SP6-mSG(J) IDT opt  CMV-SP6-mSG(J) IDT opt        Amp   
 2      pDQM005           ef1a-tdmSG(J)           ef1a-tdmSG(J)        Amp   
 3      pDQM006  ef1a-tdmSG-syntrons(J)  ef1a-tdmSG-syntrons(J)        Amp   
 4      pDQM007  ef1a-tdmSG-syntrons(J)  ef1a-tdmSG-syntrons(J)        Amp   
 
                                        plasmid_notes  \
 0  IDT optimized, for mRNA production with SP6 - ...   
 1  IDT optimized, for mRNA production with SP6 - ...   
 2     iCodon optimized for tol2 insertions - clone 4   
 3  each mStayGold contains a 51bp syntron to boos...   
 4  each mStayGold contains a 51bp syntron to boos...   
 
    used_for_injection_plasmid used_for_injection_rna  \
 0                           1                  FALSE   
 1                           1                  FALSE   
 2             

In [99]:
# CELL 3: Build constructs mapping (plasmid_base_code -> fluor/tag/tag_pos)

cons_small = (
    df_constructs[["plasmid_code", "fluor_code", "tag_code", "tag_pos"]]
    .rename(columns={"plasmid_code": "plasmid_base_code"})
    .copy()
)
cons_small["plasmid_base_code"] = cons_small["plasmid_base_code"].astype(str).str.strip()

cons_small.head()

,plasmid_base_code,fluor_code,tag_code,tag_pos
0,pDQM001,mSG,NaN,NaN
1,pDQM002,mSG,NaN,NaN
2,pDQM005,tdmSG,NaN,NaN
3,pDQM006,tdmSG,NaN,NaN
4,pDQM007,tdmSG,NaN,NaN


In [100]:
# CELL 4: Helpers for splitting code lists and building fusion labels

import pandas as pd

def split_codes(val: object) -> list[str]:
    if not isinstance(val, str) or not val.strip():
        return []
    return [c.strip() for c in val.split(",") if c.strip()]

def agg_uniq(series: pd.Series) -> str | None:
    vals = []
    for x in series:
        if pd.isna(x):
            continue
        s = str(x).strip()
        if not s:
            continue
        vals.append(s)
    if not vals:
        return None
    uniq = sorted(set(" ".join(s.split()) for s in vals))
    return ",".join(uniq)

def make_fusion_label(row: pd.Series) -> str | None:
    def norm(val: object) -> str:
        if pd.isna(val):
            return ""
        return str(val).strip()

    fluor = norm(row.get("fluor_code"))
    tag   = norm(row.get("tag_code"))
    pos   = norm(row.get("tag_pos"))

    if not fluor and not tag:
        return None
    if fluor and not tag:
        return fluor
    if not fluor and tag:
        return tag
    if fluor and tag and pos:
        return f"{fluor}::{tag}({pos})"
    return f"{fluor}::{tag}"

In [101]:
# CELL 5: Build genotype fusions from genotype_base_codes (v5)

gexp = (
    df[["roi_dir", "genotype_base_codes"]]
    .dropna(subset=["genotype_base_codes"])
    .assign(code_list=lambda d: d["genotype_base_codes"].apply(split_codes))
    .explode("code_list")
    .rename(columns={"code_list": "plasmid_base_code"})
)

gexp["plasmid_base_code"] = gexp["plasmid_base_code"].astype(str).str.strip()

gjoin = gexp.merge(cons_small, how="left", on="plasmid_base_code")

gjoin["fusion_label"] = gjoin.apply(make_fusion_label, axis=1)

g_per_roi = (
    gjoin.groupby("roi_dir", as_index=False)
    .agg(genotype_marker_fusion_labels=("fusion_label", agg_uniq))
)

df = df.merge(g_per_roi, how="left", on="roi_dir")

df[[
    "roi_dir",
    "genotype_base_codes",
    "genotype_marker_fluor_codes",
    "genotype_marker_tag_codes",
    "genotype_marker_fusion_labels",
]].head(20)

,roi_dir,genotype_base_codes,genotype_marker_fluor_codes,genotype_marker_tag_codes,genotype_marker_fusion_labels
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM005,tdmSG,NaN,tdmSG
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM005,tdmSG,NaN,tdmSG
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,NaN,None
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,NaN,None
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,NaN,None
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,NaN,None
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,NaN,None
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,NaN,None
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,NaN,None
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,NaN,None


In [102]:
# CELL 6: Build treatment fusions from treatment base codes (v5)

def collect_all_treatment_codes(row: pd.Series) -> list[str]:
    codes: list[str] = []
    codes.extend(split_codes(row.get("treatment_plasmid_base_codes")))
    codes.extend(split_codes(row.get("treatment_rna_base_codes")))
    seen = set()
    out: list[str] = []
    for c in codes:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

texp = (
    df[["roi_dir", "treatment_plasmid_base_codes", "treatment_rna_base_codes"]]
    .assign(code_list=lambda d: d.apply(collect_all_treatment_codes, axis=1))
    .explode("code_list")
)

texp = texp[texp["code_list"].notna() & (texp["code_list"].astype(str).str.strip() != "")]
texp = texp.rename(columns={"code_list": "plasmid_base_code"})
texp["plasmid_base_code"] = texp["plasmid_base_code"].astype(str).str.strip()

tjoin = texp.merge(cons_small, how="left", on="plasmid_base_code")

tjoin["fusion_label"] = tjoin.apply(make_fusion_label, axis=1)

t_per_roi = (
    tjoin.groupby("roi_dir", as_index=False)
    .agg(treatment_marker_fusion_labels=("fusion_label", agg_uniq))
)

df = df.merge(t_per_roi, how="left", on="roi_dir")

df[[
    "roi_dir",
    "treatment_plasmid_base_codes",
    "treatment_rna_base_codes",
    "treatment_marker_fluor_codes",
    "treatment_marker_tag_codes",
    "treatment_marker_fusion_labels",
]].head(20)

,roi_dir,treatment_plasmid_base_codes,treatment_rna_base_codes,treatment_marker_fluor_codes,treatment_marker_tag_codes,treatment_marker_fusion_labels
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,pDQM117,mScarlet3S2,H2B,mScarlet3S2::H2B(N)
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,pDQM117,mScarlet3S2,H2B,mScarlet3S2::H2B(N)
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,NaN,mGold2s,NaN,mGold2s
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,NaN,mGold2s,NaN,mGold2s
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,NaN,mGold2s,NaN,mGold2s
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,NaN,mGold2s,NaN,mGold2s
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,NaN,mGold2s,NaN,mGold2s
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,NaN,mGold2s,NaN,mGold2s
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,NaN,mGold2s,NaN,mGold2s
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,NaN,mGold2s,NaN,mGold2s


In [103]:
# CELL 7: Load tags and build tag_code -> localization mapping (v5)

AUTOLOAD = BASE / "seed_kits" / "2025-11-15-121231-autoload"
tags_path = AUTOLOAD / "tags.xlsx"

df_tags = pd.read_excel(tags_path)

# Normalize to a simple mapping: tag_code -> localization
# Adjust these column names if your tags.xlsx uses different headers
if "tag_code" in df_tags.columns:
    tag_code_col = "tag_code"
elif "nickname" in df_tags.columns:
    tag_code_col = "nickname"
else:
    raise RuntimeError(f"Don't see a 'tag_code' or 'nickname' column in {tags_path}")

loc_cols = [c for c in df_tags.columns if c.lower() == "localization"]
if not loc_cols:
    raise RuntimeError(f"Don't see a 'localization' column in {tags_path}")
loc_col = loc_cols[0]

tags_map = df_tags[[tag_code_col, loc_col]].copy()
tags_map.columns = ["tag_code", "localization"]
tags_map["tag_code"] = tags_map["tag_code"].astype(str).str.strip()

print("df_tags:", df_tags.shape)
tags_map.head()

df_tags: (10, 5)


,tag_code,localization
0,2Xcox8A,mitochondria
1,2xLynk,membrane
2,DHB,cell cycle
3,sec61b,ER
4,LAMP1,lysosome


In [104]:
# CELL 8: Build genotype/treatment localizations per ROI (v5)

rows = []

for _, row in df.iterrows():
    roi = row["roi_dir"]
    for source, col in [
        ("genotype", "genotype_marker_tag_codes"),
        ("treatment", "treatment_marker_tag_codes"),
    ]:
        val = row.get(col)
        if val is None or pd.isna(val):
            continue
        for tag in split_codes(val):
            rows.append({"roi_dir": roi, "source": source, "tag_code": tag})

df_tags_exp = pd.DataFrame(rows)
print("df_tags_exp:", df_tags_exp.shape)

if not df_tags_exp.empty:
    df_tags_exp = df_tags_exp.merge(tags_map, how="left", on="tag_code")

    # aggregate localizations per ROI/source
    loc_per_roi = (
        df_tags_exp.groupby(["roi_dir", "source"], as_index=False)
        .agg(localizations=("localization", agg_uniq))
    )

    # pivot into genotype vs treatment localization columns
    loc_wide = loc_per_roi.pivot(index="roi_dir", columns="source", values="localizations")
    loc_wide = loc_wide.rename(
        columns={
            "genotype": "genotype_marker_localizations",
            "treatment": "treatment_marker_localizations",
        }
    ).reset_index()

    df = df.merge(loc_wide, how="left", on="roi_dir")

df[[
    "roi_dir",
    "genotype_marker_tag_codes",
    "genotype_marker_localizations",
    "treatment_marker_tag_codes",
    "treatment_marker_localizations",
]].head(20)

df_tags_exp: (890, 3)


,roi_dir,genotype_marker_tag_codes,genotype_marker_localizations,treatment_marker_tag_codes,treatment_marker_localizations
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,NaN,H2B,histone
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,NaN,H2B,histone
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,NaN,NaN,NaN
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,NaN,NaN,NaN
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,NaN,NaN,NaN
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,NaN,NaN,NaN
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,NaN,NaN,NaN
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,NaN,NaN,NaN
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,NaN,NaN,NaN
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,NaN,NaN,NaN,NaN


In [105]:
# CELL 9: Build v5 annotations and write imaging_roi_annotations_AUTO_v5.csv

annot_cols = [
    "roi_dir",
    "parent_female", "parent_male",
    "genotype_pretty",
    "genotype_base_codes", "genotype_allele_codes",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes",
    "genotype_marker_fusion_labels",
    "genotype_marker_localizations",
    "treatment_plasmid_base_codes", "treatment_rna_base_codes",
    "treatment_marker_fluor_codes", "treatment_marker_tag_codes",
    "treatment_marker_fusion_labels",
    "treatment_marker_localizations",
    "all_marker_fluor_codes",
    "plate_id_filled", "slot_id_filled",
    "roi_index_within_slot", "roi_code",
]

annot_cols = [c for c in annot_cols if c in df.columns]

annot_v5 = df[annot_cols].copy()

annot_path_v5 = out_csv("imaging_roi_annotations_AUTO")
annot_v5.to_csv(annot_path_v5, index=False)

annot_path_v5, annot_v5.shape

(PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/imaging_roi_annotations_AUTO_v5.csv'),
 (976, 21))

In [106]:
# CELL 10: Build fluor-localization rollups from fusions + tags (v5)

def fluor_loc_label(row: pd.Series) -> str | None:
    def norm(val: object) -> str:
        if pd.isna(val):
            return ""
        return str(val).strip()

    fluor = norm(row.get("fluor_code"))
    loc   = norm(row.get("localization"))

    if not fluor:
        return None
    if loc:
        return f"{fluor}({loc})"
    return fluor

# --- genotype: base_code -> cons_small -> tags_map -> fluor(loc) ---

# gjoin has columns: roi_dir, plasmid_base_code, fluor_code, tag_code, tag_pos, ...
gjoin_loc = gjoin.merge(tags_map, how="left", on="tag_code")
gjoin_loc["fluor_loc"] = gjoin_loc.apply(fluor_loc_label, axis=1)

g_loc_per_roi = (
    gjoin_loc.groupby("roi_dir", as_index=False)
    .agg(genotype_marker_fluor_loc_labels=("fluor_loc", agg_uniq))
)

df = df.merge(g_loc_per_roi, how="left", on="roi_dir")

# --- treatment: same idea using tjoin ---

tjoin_loc = tjoin.merge(tags_map, how="left", on="tag_code")
tjoin_loc["fluor_loc"] = tjoin_loc.apply(fluor_loc_label, axis=1)

t_loc_per_roi = (
    tjoin_loc.groupby("roi_dir", as_index=False)
    .agg(treatment_marker_fluor_loc_labels=("fluor_loc", agg_uniq))
)

df = df.merge(t_loc_per_roi, how="left", on="roi_dir")

df[[
    "roi_dir",
    "genotype_marker_fluor_codes",
    "genotype_marker_tag_codes",
    "genotype_marker_localizations",
    "genotype_marker_fluor_loc_labels",
    "treatment_marker_fluor_codes",
    "treatment_marker_tag_codes",
    "treatment_marker_localizations",
    "treatment_marker_fluor_loc_labels",
]].head(20)

,roi_dir,genotype_marker_fluor_codes,genotype_marker_tag_codes,genotype_marker_localizations,genotype_marker_fluor_loc_labels,treatment_marker_fluor_codes,treatment_marker_tag_codes,treatment_marker_localizations,treatment_marker_fluor_loc_labels
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,tdmSG,NaN,NaN,tdmSG,mScarlet3S2,H2B,histone,mScarlet3S2(histone)
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,tdmSG,NaN,NaN,tdmSG,mScarlet3S2,H2B,histone,mScarlet3S2(histone)
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Halo,NaN,NaN,None,mGold2s,NaN,NaN,mGold2s
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Halo,NaN,NaN,None,mGold2s,NaN,NaN,mGold2s
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Halo,NaN,NaN,None,mGold2s,NaN,NaN,mGold2s
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Halo,NaN,NaN,None,mGold2s,NaN,NaN,mGold2s
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Halo,NaN,NaN,None,mGold2s,NaN,NaN,mGold2s
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Halo,NaN,NaN,None,mGold2s,NaN,NaN,mGold2s
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Halo,NaN,NaN,None,mGold2s,NaN,NaN,mGold2s
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Halo,NaN,NaN,None,mGold2s,NaN,NaN,mGold2s


In [107]:
# CELL: Build fusion(localization) rollups per ROI (v5)

def fusion_loc_label(row: pd.Series) -> str | None:
    # reuse make_fusion_label for fusion string
    fusion = make_fusion_label(row)
    if not fusion:
        return None

    # localization is from tags_map via merge
    loc = row.get("localization")
    if pd.isna(loc) or str(loc).strip() == "":
        return fusion
    return f"{fusion}({str(loc).strip()})"

# --- genotype fusion(localization) ---

# gjoin: roi_dir, plasmid_base_code, fluor_code, tag_code, tag_pos, ...
gjoin_loc = gjoin.merge(tags_map, how="left", on="tag_code")
gjoin_loc["fusion_loc"] = gjoin_loc.apply(fusion_loc_label, axis=1)

g_fusion_loc_per_roi = (
    gjoin_loc.groupby("roi_dir", as_index=False)
    .agg(genotype_marker_fusion_loc_labels=("fusion_loc", agg_uniq))
)

df = df.merge(g_fusion_loc_per_roi, how="left", on="roi_dir")

# --- treatment fusion(localization) ---

tjoin_loc = tjoin.merge(tags_map, how="left", on="tag_code")
tjoin_loc["fusion_loc"] = tjoin_loc.apply(fusion_loc_label, axis=1)

t_fusion_loc_per_roi = (
    tjoin_loc.groupby("roi_dir", as_index=False)
    .agg(treatment_marker_fusion_loc_labels=("fusion_loc", agg_uniq))
)

df = df.merge(t_fusion_loc_per_roi, how="left", on="roi_dir")

df[[
    "roi_dir",
    "genotype_marker_fusion_labels",
    "genotype_marker_localizations",
    "genotype_marker_fusion_loc_labels",
    "treatment_marker_fusion_labels",
    "treatment_marker_localizations",
    "treatment_marker_fusion_loc_labels",
]].head(20)

,roi_dir,genotype_marker_fusion_labels,genotype_marker_localizations,genotype_marker_fusion_loc_labels,treatment_marker_fusion_labels,treatment_marker_localizations,treatment_marker_fusion_loc_labels
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,tdmSG,NaN,tdmSG,mScarlet3S2::H2B(N),histone,mScarlet3S2::H2B(N)(histone)
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,tdmSG,NaN,tdmSG,mScarlet3S2::H2B(N),histone,mScarlet3S2::H2B(N)(histone)
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,NaN,None,mGold2s,NaN,mGold2s
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,NaN,None,mGold2s,NaN,mGold2s
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,NaN,None,mGold2s,NaN,mGold2s
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,NaN,None,mGold2s,NaN,mGold2s
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,NaN,None,mGold2s,NaN,mGold2s
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,NaN,None,mGold2s,NaN,mGold2s
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,NaN,None,mGold2s,NaN,mGold2s
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,NaN,None,mGold2s,NaN,mGold2s


In [108]:
# CELL: normalize genotype_pretty to avoid weird symbols (v5)

import re

def clean_genotype_pretty(val: object) -> str | None:
    if val is None or pd.isna(val):
        return None
    s = str(val)

    # Replace Unicode multiplication sign with ASCII " x "
    s = s.replace("×", " x ")

    # Normalize non-breaking spaces, odd spacing
    s = s.replace("\u00a0", " ")  # non-breaking space --> normal space
    s = re.sub(r"\s+", " ", s).strip()

    return s

df["genotype_pretty"] = df["genotype_pretty"].apply(clean_genotype_pretty)

df[["genotype_pretty"]].head(20)

,genotype_pretty
0,ef1a:2xLynk:tdmSG(J) (F2 of allele 301) x ef1a...
1,ef1a:2xLynk:tdmSG(J) (F2 of allele 301) x ef1a...
2,membrane Halo x membrane Halo
3,membrane Halo x membrane Halo
4,membrane Halo x membrane Halo
5,membrane Halo x membrane Halo
6,membrane Halo x membrane Halo
7,membrane Halo x membrane Halo
8,membrane Halo x membrane Halo
9,membrane Halo x membrane Halo


In [ ]:
# CELL: Infer anatomy (imaged locations / targets) per experiment slug (v5)

import pandas as pd
import re

# Path to imaging_sheet
sheet_path = BASE / "seed_kits" / "legacy_wrangling" / "raw" / "2025-11-13-124226-imaging_sheet.xlsx"
df_sheet = pd.read_excel(sheet_path)

print("df_sheet:", df_sheet.shape)

# normalize Data location and compute sheet_slug similar to slug_infer
df_sheet["Data location"] = df_sheet["Data location"].astype(str).str.strip()

def sheet_slug_from_data_location(path: str) -> str:
    p = path.replace("\\", "/").rstrip("/")
    parts = p.split("/")
    # Example: X:/abcabc/Korra_Foundation/20251110_mem-histone
    if len(parts) >= 3:
        foundation = parts[-2]  # Korra_Foundation
        experiment = parts[-1]  # 20251110_mem-histone
        return f"{foundation}/{experiment}"
    return p

df_sheet["sheet_slug"] = df_sheet["Data location"].apply(sheet_slug_from_data_location)

# clean Imaged Locations / Unique Targets text
def clean_list_text(val: object) -> str | None:
    if val is None or pd.isna(val):
        return None
    s = str(val)
    # normalize weird spaces and non-breaking spaces
    s = s.replace("\u00a0", " ")
    # collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else None

for col in ["Imaged Locations", "Unique Targets", "Unique Targets with blanks"]:
    if col in df_sheet.columns:
        df_sheet[col] = df_sheet[col].apply(clean_list_text)

# aggregate anatomy info per sheet_slug
def agg_uniq(series: pd.Series) -> str | None:
    vals = []
    for x in series:
        if x is None or pd.isna(x):
            continue
        s = str(x).strip()
        if not s:
            continue
        vals.append(s)
    if not vals:
        return None
    uniq = sorted(set(vals))
    return "; ".join(uniq)

anatomy_agg = (
    df_sheet.groupby("sheet_slug", as_index=False)
            .agg(
                anatomy_imaged_locations=("Imaged Locations", agg_uniq),
                anatomy_targets=("Unique Targets", agg_uniq),
            )
)

print("anatomy_agg:", anatomy_agg.shape)
anatomy_agg.head()

In [109]:
# CELL: Final v5 output — write imaging_roi_annotations_AUTO_v5.csv

annot_cols = [
    "roi_dir",
    "parent_female", "parent_male",
    "genotype_pretty",
    "genotype_base_codes", "genotype_allele_codes",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes",
    "genotype_marker_fusion_labels",
    "genotype_marker_localizations",
    "genotype_marker_fluor_loc_labels",
    "genotype_marker_fusion_loc_labels",
    "treatment_plasmid_base_codes", "treatment_rna_base_codes",
    "treatment_marker_fluor_codes", "treatment_marker_tag_codes",
    "treatment_marker_fusion_labels",
    "treatment_marker_localizations",
    "treatment_marker_fluor_loc_labels",
    "treatment_marker_fusion_loc_labels",
    "all_marker_fluor_codes",
    "plate_id_filled", "slot_id_filled",
    "roi_index_within_slot", "roi_code",
]

# keep only columns that actually exist (defensive)
annot_cols = [c for c in annot_cols if c in df.columns]

annot_v5 = df[annot_cols].copy()

annot_path_v5 = out_csv("imaging_roi_annotations_AUTO")
annot_v5.to_csv(annot_path_v5, index=False)

print("Wrote:", annot_path_v5)
print("Shape:", annot_v5.shape)
annot_v5.head()

Wrote: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/imaging_roi_annotations_AUTO_v5.csv
Shape: (976, 25)


,roi_dir,parent_female,parent_male,genotype_pretty,genotype_base_codes,genotype_allele_codes,genotype_marker_fluor_codes,genotype_marker_tag_codes,genotype_marker_fusion_labels,genotype_marker_localizations,...,treatment_marker_tag_codes,treatment_marker_fusion_labels,treatment_marker_localizations,treatment_marker_fluor_loc_labels,treatment_marker_fusion_loc_labels,all_marker_fluor_codes,plate_id_filled,slot_id_filled,roi_index_within_slot,roi_code
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301) x ef1a...,pDQM005,"pDQM005:301,pDQM005:301",tdmSG,NaN,tdmSG,NaN,...,H2B,mScarlet3S2::H2B(N),histone,mScarlet3S2(histone),mScarlet3S2::H2B(N)(histone),"mScarlet3S2,tdmSG",20250428-plate1,20250428-plate1-slot1,1.0,20250428-plate1-slot1-roi01
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301) x ef1a...,pDQM005,"pDQM005:301,pDQM005:301",tdmSG,NaN,tdmSG,NaN,...,H2B,mScarlet3S2::H2B(N),histone,mScarlet3S2(histone),mScarlet3S2::H2B(N)(histone),"mScarlet3S2,tdmSG",20250428-plate1,20250428-plate1-slot1,2.0,20250428-plate1-slot1-roi02
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,membrane Halo x membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",Halo,NaN,None,NaN,...,NaN,mGold2s,NaN,mGold2s,mGold2s,"Halo,mGold2s",20250501-plate1,20250501-plate1-slot1,1.0,20250501-plate1-slot1-roi01
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,membrane Halo x membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",Halo,NaN,None,NaN,...,NaN,mGold2s,NaN,mGold2s,mGold2s,"Halo,mGold2s",20250501-plate1,20250501-plate1-slot1,2.0,20250501-plate1-slot1-roi02
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,membrane Halo x membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",Halo,NaN,None,NaN,...,NaN,mGold2s,NaN,mGold2s,mGold2s,"Halo,mGold2s",20250501-plate1,20250501-plate1-slot1,3.0,20250501-plate1-slot1-roi03


In [110]:
# DIAGNOSTIC: which slugs exist in v5 and which exist in the mapper?

def extract_slug(path: str) -> str:
    p = path.replace("\\", "/").rstrip("/")
    parts = p.split("/")
    # using last two parts is working well for your data
    return "/".join(parts[-3:-1]) if len(parts) >= 3 else p

# compute slugs for v5
df["slug"] = df["roi_dir"].astype(str).apply(extract_slug)

# read the mapper
mapper_path = BASE / "seed_kits" / "legacy_wrangling" / "raw" / "roi_missing_parents_for_manual_mapping_DQM_CNH (1).csv"
df_map = pd.read_csv(mapper_path)
df_map["slug"] = df_map["slug"].astype(str).str.strip()

# compare slug sets
slugs_v5 = set(df["slug"].unique())
slugs_mapper = set(df_map["slug"].unique())

print("Total slugs in v5:", len(slugs_v5))
print("Total slugs in mapper:", len(slugs_mapper))

print("\nSlugs present in mapper but missing in v5:")
print(sorted(slugs_mapper - slugs_v5))

print("\nSlugs present in v5 but missing in mapper (these cannot be filled by mapper):")
print(sorted(slugs_v5 - slugs_mapper)[:50])   # show only first 50

Total slugs in v5: 150
Total slugs in mapper: 12

Slugs present in mapper but missing in v5:
['er-mSG_mem-mChilada', 'mem-kinetocore', 'mem-mchilada_er-mSG', 'mem_histone', 'mem_mito', 'mem_tester', 'mitomSG', 'mrna_mSG_organelle_LLS-SIM/er_roi1', 'nan', 'peroxi', 'skittes', 'skittlez']

Slugs present in v5 but missing in mapper (these cannot be filled by mapper):
['20250428_mem_histone/fish1_72hpf', '20250429_mem_cytosol/fish1_24hpf', '20250429_mem_cytosol/fish2_48hpf', '20250429_mem_cytosol/fish3_mem-halo_24hpf', '20250429_mem_cytosol/fish4_mem-halo_48hpf', '20250513_skittles/fish1', '20250513_skittles/fish10', '20250513_skittles/fish11', '20250513_skittles/fish12', '20250513_skittles/fish13', '20250513_skittles/fish14', '20250513_skittles/fish15', '20250513_skittles/fish16', '20250513_skittles/fish2', '20250513_skittles/fish3', '20250513_skittles/fish4', '20250513_skittles/fish5', '20250513_skittles/fish6', '20250513_skittles/fish7', '20250513_skittles/fish8', '20250513_skittles/fis

In [111]:
# DIAGNOSTIC: for each slug present in the mapper, count holes

holes = []

for slug in sorted(slugs_v5 & slugs_mapper):
    sub = df[df["slug"] == slug]

    missing_treatment = sub["treatment_plasmid_base_codes"].isna() | (sub["treatment_plasmid_base_codes"] == "")
    n_missing = missing_treatment.sum()

    holes.append((slug, len(sub), n_missing))

# show only slugs where mapper SHOULD fill but didn't
[x for x in holes if x[2] > 0][:20]

[]